# Patterns of Crime in Melbourne, Victoria

## KGG375 Geospatial Analysis

This project investigates the spatial distribution of crime across Melbourne suburbs and explores relationships between crime patterns and population characteristics using geospatial analysis techniques.

Authors: Shaun Knight, Kalvin Pitt
Date: October 2026

## Study Area

The study area consists of suburbs within the Melbourne metropolitan region in Victoria, Australia. Suburb boundary data obtained from Vicmap is used as the spatial framework for analysis.

The aim of this project is to examine the geographical distribution of crime and identify whether spatial clustering exists between neighbouring suburbs.

## Data Acquisition and Preparation

Three datasets were used in this project:

1. Vicmap suburb boundary polygons.
2. Victorian crime statistics.
3. Suburb population data.

These datasets were imported into Python and prepared for subsequent spatial analysis.

## Suburb Boundary Dataset

The suburb boundary dataset was loaded as a GeoDataFrame using GeoPandas. Each polygon represents a Victorian suburb and contains identifying attributes including suburb names and administrative information.

In [13]:
import geopandas as gpd
import pandas as pd

suburbs = gpd.read_file(
    r"Data/VICMapSuburbPolys/mga2020_55/esrishape/user_polygon/user_polygon-0/VMADMIN/AD_LOCALITY_AREA_POLYGON.shp"
)

print(suburbs.head())
print(suburbs.columns)

        UFI      PFI FTYPE_CODE             NAME  NFEAT_ID        NUNIQ_LOC  \
0  69885514  4160577   locality         WANTIRNA       815         WANTIRNA   
1  74680597  4160842   locality           ELTHAM       311           ELTHAM   
2  70741707  4160890   locality  SCOTCHMANS LEAD     16105  SCOTCHMANS LEAD   
3  76275819  4160639   locality         RICHMOND       664         RICHMOND   
4  76275821  4160618   locality    HAWTHORN EAST       392    HAWTHORN EAST   

  STATE  FQID  TASK_ID CRDATE_PFI  SUPER_PFI CRDATE_UFI  LABEL_USE_  \
0   VIC  4712        4 2002-01-14          0 2023-11-30           5   
1   VIC  4712     5043 2002-01-14          0 2025-10-30           5   
2   VIC  4897        4 2002-01-14          0 2024-06-05           5   
3   VIC  4712     5102 2002-01-14          0 2026-04-27           5   
4   VIC  4712     5102 2002-01-14          0 2026-04-27           5   

                                            geometry  
0  POLYGON ((344167.972 5810514.524, 344176

## Population Dataset

Population data was obtained for Victorian suburbs and used to standardise crime counts into population-adjusted crime rates. Standardisation allows meaningful comparison between suburbs of different sizes.

In [14]:
pop = pd.read_csv("Data/Suburb_pop.csv")
crime = pd.read_csv("Data/Crime_rates.csv")

print(pop.columns)
print(crime.columns)

Index(['SAL CODE', 'SUBURB NAME', 'TOT POP', 'Unnamed: 3', 'Unnamed: 4'], dtype='str')
Index(['Year', 'Year ending', 'Local Government Area', 'Postcode',
       'Suburb/Town Name', 'Offence Division', 'Offence Subdivision',
       'Offence Subgroup', ' Offence Count '],
      dtype='str')


## Crime Dataset

Crime data were obtained from the Victorian Crime Statistics Agency. The selected table contains offence counts recorded at the suburb or town level and includes offence classifications, local government areas, and crime counts.

Suburb-level data were selected because they provide a finer spatial resolution than local government area summaries and are therefore more suitable for spatial pattern analysis.

In [15]:
crime = pd.read_csv("Data/Crime_rates.csv",)

crime.head()

,Year,Year ending,Local Government Area,Postcode,Suburb/Town Name,Offence Division,Offence Subdivision,Offence Subgroup,Offence Count
0,2026,March,Alpine,3691,Coral Bank,A Crimes against the person,A20 Assault and related offences,A211 FV Serious assault,1
1,2026,March,Alpine,3691,Coral Bank,B Property and deception offences,B20 Property damage,B21 Criminal damage,1
2,2026,March,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B42 Steal from a motor vehicle,1
3,2026,March,Alpine,3691,Dederang,B Property and deception offences,B40 Theft,B49 Other theft,1
4,2026,March,Alpine,3691,Dederang,C Drug offences,C30 Drug use and possession,C32 Drug possession,1


## Data Cleaning and Harmonisation

The crime, population, and boundary datasets were sourced from different organisations and therefore used different geographic definitions and naming conventions.

Prior to analysis, suburb names were standardised to ensure consistency between datasets. Records that could not be matched across all datasets were excluded from further analysis. Only suburbs present in the crime, population, and boundary datasets were retained for subsequent spatial analysis.

In [25]:
suburbs["NAME"] = (
    suburbs["NAME"]
    .str.upper()
    .str.strip()
)

pop["SUBURB NAME"] = (
    pop["SUBURB NAME"]
    .str.upper()
    .str.replace(" (VIC.)", "", regex=False)
    .str.strip()
)

crime["Suburb/Town Name"] = (
    crime["Suburb/Town Name"]
    .str.upper()
    .str.strip()
)

crime_suburbs = set(crime["Suburb/Town Name"].dropna())
pop_suburbs = set(pop["SUBURB NAME"].dropna())
poly_suburbs = set(suburbs["NAME"].dropna())

print("Crime suburbs:", len(crime_suburbs))
print("Population suburbs:", len(pop_suburbs))
print("Polygon suburbs:", len(poly_suburbs))

all_three = crime_suburbs & pop_suburbs & poly_suburbs

print("Matching suburbs:", len(all_three))

Crime suburbs: 2867
Population suburbs: 2944
Polygon suburbs: 1274
Matching suburbs: 1251


## Matching Geographic Units

After standardising suburb names, the overlap between datasets was assessed.

A total of 1,251 suburbs were present in all three datasets and were retained for analysis. Because spatial analysis requires both attribute and boundary information, the common set of suburbs was used as the final study dataset.

Suburbs without matching records across all datasets were excluded from subsequent analysis.

In [27]:
crime = crime[
    crime["Suburb/Town Name"].isin(all_three)
]

pop = pop[
    pop["SUBURB NAME"].isin(all_three)
]

suburbs = suburbs[
    suburbs["NAME"].isin(all_three)
]

print(len(crime))
print(len(pop))
print(len(suburbs))

266445
1251
1251


## Aggregation of Crime Data

The crime dataset contained individual offence categories and subcategories for each suburb. To enable suburb-level spatial analysis, offence counts were aggregated to produce a single total crime count for each suburb.

This aggregation ensured that each suburb was represented by one crime value that could be directly compared with population and spatial boundary datasets.

In [28]:
crime["Offence Count"] = pd.to_numeric(
    crime["Offence Count"],
    errors="coerce"
)

crime_summary = (
    crime
    .groupby("Suburb/Town Name", as_index=False)
    ["Offence Count"]
    .sum()
)

crime_summary.columns = [
    "SUBURB",
    "TOTAL_CRIME"
]

crime_summary.head()

KeyError: 'Offence Count'